# Formal Metrics Overview

            读取 formal A/B/C 的 `eval_metrics_best.json` 和 pretrain metrics，回答：

            - 强 pretrain 的 ceiling 是否正常？
            - A/B/C 在每个 MedMNIST 数据集上的 ACC/AUC/Macro-F1/Balanced ACC 如何？
            - 是否存在 ACC 高但 Macro-F1 或 Balanced ACC 低的类别不均衡风险？


In [ ]:

from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

EXP_ROOT = Path("/data/zengqiang/experiments/ncfm_medmnist_ablation_20260519")

def require_exp_root():
    if not EXP_ROOT.exists():
        raise FileNotFoundError(
            f"EXP_ROOT not found: {EXP_ROOT}. "
            "Edit EXP_ROOT in the first code cell to your experiment directory."
        )

def ensure_report_dir(*parts):
    path = EXP_ROOT / "reports" / "cam" / Path(*parts)
    path.mkdir(parents=True, exist_ok=True)
    return path

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_group(name):
    if name == "real_train":
        return "real_train"
    if name.startswith("ipc10_"):
        return name[len("ipc10_"):]
    return name

def resolve_result_path(value):
    p = Path(str(value))
    if p.exists():
        return p
    if str(value).startswith("/"):
        return p
    q = EXP_ROOT / value
    return q

def load_eval_metrics():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "runs").glob("*/ipc10/*/eval_metrics_best.json")):
        item = read_json(path)
        item["dataset"] = path.parents[2].name
        item["group"] = path.parent.name
        item["metrics_path"] = str(path)
        rows.append(item)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    order = {"A_pure_ncfd_wopsi": 0, "B_minmax_ncfm_psi": 1, "C_code_default_enhanced": 2}
    df["_order"] = df["group"].map(order).fillna(99)
    return df.sort_values(["dataset", "_order"]).drop(columns=["_order"])

def load_cam_summaries():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "results" / "cam").glob("*/*/summary.csv")):
        dataset = path.parents[1].name
        group = normalize_group(path.parent.name)
        df = pd.read_csv(path)
        if df.empty:
            continue
        df["dataset"] = dataset
        df["group"] = group
        df["summary_path"] = str(path)
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    for col in ["index", "y_true", "y_pred", "confidence", "correct", "cam_entropy", "topk_activation_ratio"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def cam_group_summary(cam_df):
    if cam_df.empty:
        return cam_df
    grouped = (
        cam_df.groupby(["dataset", "group"], as_index=False)
        .agg(
            n=("index", "count"),
            cam_acc=("correct", "mean"),
            mean_confidence=("confidence", "mean"),
            mean_entropy=("cam_entropy", "mean"),
            mean_top10_mass=("topk_activation_ratio", "mean"),
            correct_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
            correct_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
        )
    )
    order = {"real_train": 0, "A_pure_ncfd_wopsi": 1, "B_minmax_ncfm_psi": 2, "C_code_default_enhanced": 3}
    grouped["_order"] = grouped["group"].map(order).fillna(99)
    return grouped.sort_values(["dataset", "_order"]).drop(columns=["_order"])


In [ ]:

eval_df = load_eval_metrics()
display(eval_df[[
    "dataset", "group", "acc_percent", "auc_macro_ovr", "macro_f1",
    "balanced_acc", "sensitivity", "specificity", "auprc", "epoch", "checkpoint_path"
]] if not eval_df.empty else eval_df)

report_dir = ensure_report_dir()
if not eval_df.empty:
    eval_df.to_csv(report_dir / "formal_eval_metrics_all.csv", index=False)


In [ ]:

metrics = ["acc_percent", "auc_macro_ovr", "macro_f1", "balanced_acc"]
if not eval_df.empty:
    for metric in metrics:
        pivot = eval_df.pivot_table(index="dataset", columns="group", values=metric, aggfunc="first")
        display(pivot)
        ax = pivot.plot(kind="bar", figsize=(9, 4), rot=0)
        ax.set_title(metric)
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()


In [ ]:

rows = []
for path in sorted((EXP_ROOT / "reports" / "pretrain").glob("*/metrics.jsonl")):
    dataset = path.parent.name
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            item["dataset"] = dataset
            rows.append(item)

pre_df = pd.DataFrame(rows)
if pre_df.empty:
    print("No pretrain metrics found.")
else:
    final_pre = pre_df.sort_values("epoch").groupby(["dataset", "model_id"], as_index=False).tail(1)
    pre_summary = final_pre.groupby("dataset").agg(
        models=("model_id", "nunique"),
        acc_mean=("acc_percent", "mean"),
        acc_std=("acc_percent", "std"),
        auc_mean=("auc_macro_ovr", "mean"),
        macro_f1_mean=("macro_f1", "mean"),
        balanced_acc_mean=("balanced_acc", "mean"),
    ).reset_index()
    display(pre_summary)
    pre_summary.to_csv(report_dir / "pretrain_final_epoch_summary.csv", index=False)


## 解读提示

            - `pretrain_final_epoch_summary.csv` 用来判断 feature extractor 是否可靠。
            - `formal_eval_metrics_all.csv` 是正式蒸馏指标总表。
            - 若某组 ACC 高但 Macro-F1 / Balanced ACC 低，应优先怀疑类别不均衡或只保留了容易类别的判别结构。
